# Pipeline RAG — Voice of Customer Intelligence

Este notebook demonstra o pipeline completo de **Retrieval-Augmented Generation (RAG)** sobre as avaliações do dataset Olist.

**Pré-requisitos:**
1. `01_exploracao.ipynb` executado → `data/processed/documents.json`
2. `02_base_conhecimento.ipynb` executado → `data/processed/reviews.index` e `documents.pkl`
3. Uma chave de LLM no arquivo `.env` na raiz do projeto — `GROQ_API_KEY` (gratuita) **ou** `OPENAI_API_KEY`

**Fluxo do pipeline:**
```
Pergunta do usuário
       ↓
Embedding da query (sentence-transformers)
       ↓
Busca semântica no índice FAISS
       ↓
Filtro por limiar de similaridade (≥ 0.30)
       ↓
Construção do contexto com as avaliações recuperadas
       ↓
Geração de resposta pelo LLM (Groq / OpenAI)
       ↓
Resposta fundamentada + evidências rastreáveis
```

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path().resolve().parent))

load_dotenv(Path("../.env"))

from src.retrieval import ReviewSearcher
from src.rag import answer, build_context

DATA_PROCESSED = Path("../data/processed")

assert (DATA_PROCESSED / "reviews.index").exists(), (
    "Índice não encontrado. Execute 02_base_conhecimento.ipynb primeiro."
)

# Provedor de LLM: usa Groq (gratuito) se GROQ_API_KEY estiver definida;
# caso contrário usa OpenAI (OPENAI_API_KEY). A API da Groq é compatível
# com o SDK da OpenAI — basta trocar base_url e api_key.
if os.getenv("GROQ_API_KEY"):
    client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
    )
    MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
    print(f"Provedor LLM: Groq  |  modelo: {MODEL}")
else:
    client = OpenAI()  # usa OPENAI_API_KEY do .env
    MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    print(f"Provedor LLM: OpenAI  |  modelo: {MODEL}")

searcher = ReviewSearcher(DATA_PROCESSED)

print(f"Índice carregado: {searcher.index.ntotal:,} avaliações indexadas")

W1106 11:46:38.978000 81897 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Provedor LLM: Groq  |  modelo: openai/gpt-oss-120b


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Índice carregado: 42,687 avaliações indexadas


## 1. Recuperação semântica

Antes de gerar qualquer resposta, vamos entender o que acontece na etapa de recuperação: a pergunta é convertida em um vetor e o FAISS retorna as avaliações mais próximas no espaço semântico.

In [2]:
query = "problemas com atraso na entrega"

results = searcher.search(query, k=5)

print(f"Query: '{query}'")
print(f"Avaliações recuperadas: {len(results)}")
print()
for i, r in enumerate(results, 1):
    meta = r.get('metadata', {})
    print(f"  [{i}] Nota: {meta.get('review_score', '?')}/5  |  Similaridade: {r['similarity_score']:.3f}")
    text = r['text']
    print(f"      {text[:200]}..." if len(text) > 200 else f"      {text}")
    print()

Query: 'problemas com atraso na entrega'
Avaliações recuperadas: 5

  [1] Nota: 5/5  |  Similaridade: 0.884
      A entrega atrasou

  [2] Nota: 1/5  |  Similaridade: 0.877
      demora na entrega

  [3] Nota: 1/5  |  Similaridade: 0.869
      Pessimo muita demora na entrega

  [4] Nota: 1/5  |  Similaridade: 0.857
      demora na entrega (ainda nao chegou fora do prazo)

  [5] Nota: 1/5  |  Similaridade: 0.853
      Muita demora na entrega



## 2. Construção do contexto

As avaliações recuperadas são formatadas em um bloco de contexto estruturado antes de serem enviadas ao LLM. Cada avaliação é numerada e acompanhada de sua nota e similaridade, permitindo rastreabilidade.

In [3]:
context = build_context(results)
print(context[:1000], "..." if len(context) > 1000 else "")

[Avaliação 1 | Nota: 5/5 | Similaridade: 0.88]
A entrega atrasou

---

[Avaliação 2 | Nota: 1/5 | Similaridade: 0.88]
demora na entrega

---

[Avaliação 3 | Nota: 1/5 | Similaridade: 0.87]
Pessimo muita demora na entrega

---

[Avaliação 4 | Nota: 1/5 | Similaridade: 0.86]
demora na entrega (ainda nao chegou fora do prazo)

---

[Avaliação 5 | Nota: 1/5 | Similaridade: 0.85]
Muita demora na entrega 


## 3. Pipeline RAG completo

A função `answer()` encapsula todo o fluxo: recuperação → filtragem → contexto → LLM.

A função auxiliar `demo_answer()` abaixo imprime a resposta e as evidências de forma legível.

In [4]:
def demo_answer(question: str, k: int = 15) -> None:
    """Executa o pipeline RAG e imprime resposta + evidências."""
    print(f"{'='*70}")
    print(f"PERGUNTA: {question}")
    print('='*70)

    result = answer(question, searcher, k=k, model=MODEL, client=client)

    print("\nRESPOSTA:")
    print(result["answer"])

    print(f"\nEVIDÊNCIAS UTILIZADAS ({len(result['evidence'])} avaliações recuperadas):")
    if not result["has_evidence"]:
        print("  (nenhuma evidência suficiente encontrada)")
    else:
        for i, doc in enumerate(result["evidence"], 1):
            meta = doc.get('metadata', {})
            print(f"  [{i}] Nota: {meta.get('review_score', '?')}/5  |  Similaridade: {doc['similarity_score']:.3f}")
            text = doc['text']
            print(f"      {text[:180]}..." if len(text) > 180 else f"      {text}")
    print()

## 4. Exemplos de perguntas

### 4.1 Investigação de reclamações

In [5]:
demo_answer("Quais são os principais problemas relatados pelos clientes em suas avaliações?")

PERGUNTA: Quais são os principais problemas relatados pelos clientes em suas avaliações?



RESPOSTA:
**Principais problemas apontados pelos clientes (com base nas avaliações fornecidas)**  

| Problema identificado | Evidências nas avaliações |
|-----------------------|---------------------------|
| **Produto diferente ou incorreto em relação ao anunciado** | • “Produto recebido divergente ao comprado” (Avaliação 1). <br>• “reclamação. produto incorreto” (Avaliação 7). <br>• “produto incorreto. O produto não está nas medidas corretas indicadas na oferta” (Avaliação 15). |
| **Defeito ou baixa qualidade do produto** | • “O produto está com problemas (defeito).” (Avaliação 9). <br>• “Produto de péssima qualidade.” (Avaliação 14). |
| **Cancelamento de pedido sem explicação** | • “compra efetuada, débito autorizado, pedido cancelado sem explicações.” (Avaliação 4). |
| **Falta de respeito e tratamento inadequado ao cliente** | • “Extrema falta de respeito e consideração com os clientes.” (Avaliação 4). <br>• “Desrespeito com o cliente. Falta de respeito com o cliente.” (Avalia

### 4.2 Tema específico — entrega

In [6]:
demo_answer("O que os clientes relatam sobre problemas relacionados à entrega?")

PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?



RESPOSTA:
**Resumo dos relatos dos clientes sobre problemas de entrega**

| Tipo de problema | Quantidade de avaliações que mencionam | Trechos representativos |
|------------------|----------------------------------------|------------------------|
| **Informação confusa ou falta de comunicação** | 1 | “muito confusa a informação sobre entrega” (Avaliação 1) |
| **Desencontros / divergências entre o que consta no sistema e a realidade** | 2 | “há divergências e problemas na entrega, consta como produto entregue, porém foi entregue apenas 1 produto” (Avaliação 3) <br> “muitos desencontros nas entregas de produtos” (Avaliação 2) |
| **Atrasos ou demora na entrega** | 9 | “Atraso na entrega” (Avaliações 5, 6, 7, 8) <br> “Demora na entrega” (Avaliações 9, 10, 11, 12) |
| **Problemas específicos com os Correios** | 3 | “Embora tenha havido problemas com a entrega pelos correios o produto chegou” (Avaliação 4) <br> “o atraso na entrega foi causado pelos serviços dos Correios” (Avaliação 13)

### 4.3 Busca por experiências positivas

In [7]:
demo_answer("Quais aspectos da experiência de compra são mais elogiados pelos clientes?")

PERGUNTA: Quais aspectos da experiência de compra são mais elogiados pelos clientes?



RESPOSTA:
**Aspectos mais elogiados pelos clientes (com base nas avaliações fornecidas)**  

| Aspecto | Evidência nas avaliações |
|---|---|
| **Entrega pontual / rapidez** | “Entrega. Foi Positiva a experiência de compra.” (Avaliação 1) <br> “Recomendo a compra nessa loja, primam pela pontualidade e qualidade dos produtos.” (Avaliação 3) <br> “não é a primeira vez que tenho à alegria de comprar na loja, e sempre me surpreendendo com a pontualidade…” (Avaliação 12) |
| **Qualidade do produto** | “Produto de qualidade!. Compromisso com o cliente e qualidade no produto resumem essa compra.” (Avaliação 5) <br> “Ótimo vendedor!. Produto de qualidade, excelência no processo de compra.” (Avaliação 6) |
| **Atendimento / comprometimento do vendedor** | “bom serviço. Satisfatória, é a palavra que define minha experiência de compra… obrigado pelo atendimento.” (Avaliação 2) <br> “Ótimo vendedor!” (Avaliação 6) <br> “Ótimo a loja em sua responsabilidade com o comprador.” (Avaliação 9) |
| **Sa

### 4.4 Investigação de insatisfação

In [8]:
demo_answer("Quais padrões podem ser identificados nas avaliações de clientes insatisfeitos?")

PERGUNTA: Quais padrões podem ser identificados nas avaliações de clientes insatisfeitos?



RESPOSTA:
**Padrões recorrentes nas avaliações de clientes insatisfeitos**

| Tema / Problema | Avaliações que evidenciam | Trechos citados |
|-----------------|---------------------------|-----------------|
| **Falta de respeito / tratamento inadequado** | 1, 7, 9, 14 | “Desrespeito com o cliente. Falta de respeito com o cliente” (Avaliação 1); “Ruim. Falta de respeito com cliente” (Avaliação 7); “Extrema falta de respeito e consideração com os clientes.” (Avaliação 9); “ainda recomendo. … Piada né!” (Avaliação 14) |
| **Qualidade do produto abaixo do esperado** | 6, 10, 11, 12, 15 | “Produto de péssima qualidade.” (Avaliação 6); “produto insatisfatório… produto de baixa qualidade” (Avaliação 10); “A qualidade do produto não ficou a contento” (Avaliação 11); “má qualidade do produto” (Avaliação 12); “produto entregue… de baixíssima qualidade” (Avaliação 15) |
| **Produto diferente do anunciado / divergência entre o que foi comprado e o que chegou** | 8, 15 | “Produto recebido diverge

### 4.5 Exploração baseada em evidências

In [9]:
demo_answer("Quais avaliações sustentam a conclusão de que existem problemas recorrentes relacionados ao prazo de entrega?")

PERGUNTA: Quais avaliações sustentam a conclusão de que existem problemas recorrentes relacionados ao prazo de entrega?



RESPOSTA:
**Conclusão:**  
As avaliações apresentadas apontam, de forma recorrente, problemas relacionados ao prazo de entrega (demora, atrasos e informações de status inconsistentes).

---

### Avaliações que evidenciam o problema de prazo de entrega

| Nº da Avaliação | Nota | Trecho que indica o problema |
|-----------------|------|------------------------------|
| Avaliação 1 | 5/5 | “Muita demora na entrega” |
| Avaliação 2 | 1/5 | “Muita demora na entrega” |
| Avaliação 3 | 1/5 | “Muita demora na entrega” |
| Avaliação 4 | 1/5 | “Muita demora na entrega” |
| Avaliação 5 | 1/5 | “Pessimo muita demora na entrega” |
| Avaliação 7 | 4/5 | “Muita demora na entrega pelos Correios.” |
| Avaliação 8 | 3/5 | “O atraso na entrega prejudicou a obra” |
| Avaliação 9 | 1/5 | “Nao tem boa avaliação por atraso na entrega” |
| Avaliação 10 | 4/5 | “O status da entrega é ruim, sempre desatualizado, mostra que teve erro na entrega, mudam o prazo e no fim entregam antes.” |
| Avaliação 11 | 2/5 | 

## 5. Tratamento de perguntas sem evidências suficientes

Quando a pergunta está fora do escopo da base de conhecimento — ou quando não existem avaliações suficientemente similares — o sistema reconhece suas limitações em vez de inventar uma resposta.

In [10]:
# Pergunta fora do escopo da base (não há avaliações sobre clima ou política)
demo_answer("Qual foi o impacto da taxa de câmbio no preço dos produtos em 2017?")

PERGUNTA: Qual foi o impacto da taxa de câmbio no preço dos produtos em 2017?



RESPOSTA:
**Resposta**

Não há informações nas avaliações fornecidas que abordem a taxa de câmbio ou seu impacto nos preços dos produtos em 2017.  

- As avaliações tratam, principalmente, de questões de entrega, estado do produto, prazo de pagamento e percepção de preço (“*atendeu o procurado preço e entrega*”, “*Produto e preço bons*”, “*O produto está de acordo com o preço*”, “*Produto bom relativo ao preço*”), mas nenhuma menciona variações cambiais, ajustes de preço decorrentes da cotação do dólar ou de outra moeda.

**Conclusão:**  
Com base nas avaliações apresentadas, não é possível responder à pergunta sobre o impacto da taxa de câmbio nos preços dos produtos em 2017. Para obter essa informação seria necessário analisar fontes que contenham dados explícitos sobre variações cambiais ou comentários dos clientes a esse respeito.

EVIDÊNCIAS UTILIZADAS (15 avaliações recuperadas):
  [1] Nota: 1/5  |  Similaridade: 0.598
      Produto não foi entregue na prevista de 19/12/2017. Co

In [11]:
# Pergunta muito específica que pode não ter evidências suficientes
demo_answer("O que os clientes pensam sobre o serviço de montagem de móveis?")

PERGUNTA: O que os clientes pensam sobre o serviço de montagem de móveis?



RESPOSTA:
**Opinião dos clientes sobre o serviço de montagem de móveis (com base nas avaliações fornecidas)**  

| Avaliação | Comentário relevante | Sentimento |
|-----------|----------------------|------------|
| **Avaliação 7** | “Maravilhosa a qualidade. Muito boa a cadeira, ótima qualidade, **e clara a explicação pra montagem**.” | Positivo – o cliente elogia a clareza das instruções de montagem. |

**Análise geral**

1. **Ponto positivo identificado**  
   - Apenas a **Avaliação 7** menciona explicitamente o processo de montagem, destacando que as instruções são claras e facilitam a montagem do produto.

2. **Ausência de informações adicionais**  
   - As demais avaliações (1 a 6, 8 a 15) não trazem nenhum comentário sobre montagem, seja elogiando, criticando ou sugerindo melhorias.  
   - Portanto, não há dados suficientes para afirmar tendências, problemas recorrentes ou outras percepções dos clientes sobre esse aspecto.

**Conclusão**

- **O que os clientes dizem:** Há pelo m

## 6. Exploração com filtros de metadados (desafio adicional)

Utilizamos os metadados enriquecidos para perguntas mais contextualizadas, filtrando as avaliações antes da busca ou incluindo informações de contexto na análise.

In [12]:
# Pergunta específica sobre avaliações negativas (nota 1)
demo_answer("Quais são os principais problemas relatados nas avaliações com nota 1 ou 2?")

PERGUNTA: Quais são os principais problemas relatados nas avaliações com nota 1 ou 2?



RESPOSTA:
**Principais problemas apontados nas avaliações com nota 1 ou 2**

| Problema identificado | Avaliações que mencionam (trecho citado) |
|-----------------------|------------------------------------------|
| **Não recebimento do material** | Avaliação 2 – “*Não posso avaliar sem ter recebido o material, até agora minha avaliação é extremamente negativa.*” |
| **Produto entregue errado / incorreto** | Avaliação 7 – “*Produto chegou errado, preciso que resolva com urgência*”  <br> Avaliação 9 – “*produto incorreto. O produto não está nas medidas corretas indicadas na oferta…*”  <br> Avaliação 14 – “*reclamação. produto incorreto*” |
| **Problemas de tamanho ou medida** | Avaliação 9 – “*O produto não está nas medidas corretas indicadas na oferta.*” |
| **Qualidade do produto considerada péssima** | Avaliação 6 – “*Produto de péssima qualidade.*” |
| **Insatisfação geral sem detalhe** | Avaliação 5 – “*insatisfação 1*” |
| **Falta de informação para avaliação** | Avaliação 3 – “

In [13]:
# Exploração por categoria de produto
demo_answer("O que os clientes relatam sobre a qualidade dos produtos de eletrônicos?")

PERGUNTA: O que os clientes relatam sobre a qualidade dos produtos de eletrônicos?



RESPOSTA:
**Resumo do que os clientes dizem sobre a qualidade dos produtos eletrônicos**

| Tipo de comentário | Avaliações que exemplificam | Sentimento predominante |
|--------------------|----------------------------|--------------------------|
| **Qualidade positiva** | - “Pontualidade e **qualidade do produto**.” (Avaliação 2) <br> - “produto **de qualidade**” (Avaliações 4, 5, 6, 7, 8, 9) <br> - “O **produto é de qualidade** e de custo benefício” (Avaliação 10) <br> - “ótimo **produto e qualidade** e atendimento top” (Avaliação 11) <br> - “Ótimo **produto, qualidade e eficiência**.” (Avaliação 13) <br> - “Tanto o **produto** quanto o atendimento é de **qualidade**.” (Avaliação 14) | Muito positivo – a maioria dos clientes destaca que o produto tem boa qualidade, muitas vezes associando‑a a bom custo‑benefício e eficiência. |
| **Qualidade negativa ou abaixo do esperado** | - “**má qualidade do produto**” (Avaliação 12) <br> - “**Qualidade do produto deixa a desejar**” (Avaliação